# 08 · El ensamble de 5 SNGP

Un ensamble es entrenar la MISMA red varias veces cambiando solo la semilla (los pesos con los que arranca, el orden en que ve los datos y las random features de la capa GP) y después promediar lo que predicen. Cada red, por ese azar, termina en un orden un poco distinto; al promediar, los errores propios de cada una se compensan (Lakshminarayanan et al., 2017). Además nos da una segunda medida de incertidumbre: la **discrepancia**, cuánto se contradicen las redes entre sí (la varianza de sus 5 logits).

Todo quedó fijado antes de ver resultados: K = 5 redes, la misma red base de siempre (128 de ancho, 2 bloques, coef 0.9, lr 1e-3) y el pool no se usa para elegir nada.

- **Lee:** `outputs/datos_procesados.npz`, `outputs/ids_*.csv`, `outputs/preds/xgboost_*.csv`.
- **Guarda:** `outputs/preds/sngp_ensamble.npz` (logits y varianzas de las 5 redes) y `outputs/ensamble_resultados.csv`.
- **En el informe:** el "Ensamble" de la Tabla 1, la Tabla 2 y las figuras 2, 4 y 5 (se calculan en el 04 y el 10 con lo que guarda este notebook), los cocientes 1,01 y 1,19 de la sección 4.1 (Bloque 5) y los terciles de discrepancia y de varianza GP (0,741 → 0,694 y 0,685 → 0,760, Bloque 5).
- **Tiempo:** ~15 minutos (5 redes de 2-4 minutos cada una).

In [1]:
# Bloque 1. Cargamos los datos de siempre y las predicciones de XGBoost (alineadas con las filas de X).
# Además del ensamble, en el Bloque 3 probamos mezclar el ensamble con XGBoost (promedio 50/50 de los logits), para ver
# si árboles y red ven cosas distintas. Esa mezcla es una prueba que no va al informe (empata con XGBoost).
import os
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from redes import SNGP, entrenar, ajustar_covarianza, predecir   # las piezas de la red, en redes.py
from conformal import conformal_select                          # Conformal Selection (p-valores + BH)

torch.set_num_threads(os.cpu_count())   # que torch use todos los núcleos de la compu (entrenar en CPU es lento)
K = 5                    # cantidad de redes del ensamble
MAX_EPOCAS = 40          # tope de épocas por red (igual corta antes por early stopping)
SEED = 42                # semilla para el bootstrap del bloque 3

datos = np.load("outputs/datos_procesados.npz", allow_pickle=True)   # las matrices que armamos en el 01
# astype(np.float32): torch trabaja con números de 32 bits; así no hay que convertir después
X_tr, y_tr = datos["X_train"].astype(np.float32), datos["y_train"].astype(np.float32)   # train (60%)
X_va, y_va = datos["X_calib"].astype(np.float32), datos["y_calib"].astype(np.float32)   # validación (10%)
X_po, y_po = datos["X_test"].astype(np.float32), datos["y_test"].astype(np.float32)     # pool (30%)
ids_va = pd.read_csv("outputs/ids_calib.csv")["SK_ID_CURR"].to_numpy()   # ID de cada fila de validación
ids_po = pd.read_csv("outputs/ids_test.csv")["SK_ID_CURR"].to_numpy()    # ID de cada fila del pool

# Predicciones de XGBoost, ordenadas igual que las filas de X: set_index pone el ID como índice y .loc[ids]
# reordena las filas para que la fila i del CSV sea el mismo solicitante que la fila i de X
xgb_va = pd.read_csv("outputs/preds/xgboost_calib.csv").set_index("SK_ID_CURR").loc[ids_va]
xgb_po = pd.read_csv("outputs/preds/xgboost_test.csv").set_index("SK_ID_CURR").loc[ids_po]
# Si el TARGET del CSV no coincide fila por fila con el de X, algo está corrido: frena todo
assert (xgb_va["TARGET"].to_numpy() == y_va).all() and (xgb_po["TARGET"].to_numpy() == y_po).all(), "desalineado"
# Vector True/False: True = el solicitante es thin-file (sin registros en el buró)
thin_va, thin_po = xgb_va["thin_file"].to_numpy().astype(bool), xgb_po["thin_file"].to_numpy().astype(bool)
print("formas:", X_tr.shape, X_va.shape, X_po.shape, "| thin pool:", int(thin_po.sum()))

# 200 variables en las tres partes y 13.206 thin-file en el pool.

formas: (184506, 200) (30751, 200) (92254, 200) | thin pool: 13206


In [2]:
# Bloque 2. Entrenamos las K redes, cada una con su semilla (1000 + k), y guardamos el logit y la varianza de cada
# una para validación y pool. Cada red tarda lo mismo que la SNGP del 05, unos 2-4 minutos. Todo queda en
# outputs/preds/sngp_ensamble.npz para no tener que repetirlo (el 04 y el 10 lo leen de ahí).
ds_tr = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
loader_cov = DataLoader(ds_tr, batch_size=4096)                  # todo train en tandas de 4096, para la covarianza
ds_va = TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va))
va = DataLoader(ds_va, batch_size=4096)                          # validación: no se mezcla, solo se evalúa

lg_va, vr_va, lg_po, vr_po = [], [], [], []      # listas vacías: logits y varianzas de cada red (val y pool)
for k in range(K):                               # k = 0, 1, 2, 3, 4: una vuelta por red
    t0 = time.time()                             # para medir cuánto tarda cada red
    torch.manual_seed(1000 + k)                                  # semilla distinta por red: otros pesos iniciales y otras random features
    gen = torch.Generator().manual_seed(1000 + k)                # mezcla de batches distinta por red
    # Loader de train: tandas de 512 solicitantes, mezcladas en cada época (shuffle) según el generador de esta red
    tr = DataLoader(ds_tr, batch_size=512, shuffle=True, generator=gen)
    red = SNGP(d_in=X_tr.shape[1])                               # la red base de siempre: 200 variables de entrada
    # Entrenar con early stopping: lr = tamaño del paso; corta si pasan 5 épocas sin mejorar en validación
    entrenar(red, tr, va, lr=1e-3, max_epocas=MAX_EPOCAS, paciencia=5, verbose=False)
    ajustar_covarianza(red, loader_cov)                          # una pasada por train: la "memoria" de dónde hubo datos
    # predecir devuelve (logit de default, varianza); los agregamos a las listas de validación y de pool
    a, b = predecir(red, torch.from_numpy(X_va)); lg_va.append(a); vr_va.append(b)
    a, b = predecir(red, torch.from_numpy(X_po)); lg_po.append(a); vr_po.append(b)
    print(f"red {k}: AUC val total {roc_auc_score(y_va, lg_va[k]):.4f} thin {roc_auc_score(y_va[thin_va], lg_va[k][thin_va]):.4f}"
          f" | {(time.time() - t0) / 60:.1f} min")

# Pasamos las listas a matrices de forma [K, n]: fila k = la red k, columna i = el solicitante i
lg_va, vr_va, lg_po, vr_po = map(np.array, (lg_va, vr_va, lg_po, vr_po))   # forma [K, n]
np.savez("outputs/preds/sngp_ensamble.npz", lg_va=lg_va, vr_va=vr_va, lg_po=lg_po, vr_po=vr_po)   # las 4 matrices en un archivo
print("guardado: outputs/preds/sngp_ensamble.npz")

# Las 5 redes dan AUC thin de validación entre 0.7167 y 0.7235: solo por cambiar la semilla, el AUC de una red se mueve
# ~0.007, que es casi la mitad de la brecha con XGBoost. Comparar contra una sola red tiene bastante ruido, y por eso
# tiene sentido el ensamble.

red 0: AUC val total 0.7567 thin 0.7175 | 3.7 min
red 1: AUC val total 0.7563 thin 0.7167 | 2.5 min
red 2: AUC val total 0.7563 thin 0.7200 | 2.6 min
red 3: AUC val total 0.7560 thin 0.7235 | 2.9 min
red 4: AUC val total 0.7568 thin 0.7185 | 2.3 min
guardado: outputs/preds/sngp_ensamble.npz


In [3]:
# Bloque 3. Armamos los modelos a comparar, todos con el logit de mora como score: XGBoost, una SNGP sola, el ensamble
# (promedio de las K redes) y las dos mezclas con XGBoost. Medimos el AUC en validación y en el pool por segmento, y
# hacemos el bootstrap pareado de la diferencia de AUC thin contra XGBoost ("pareado" = cada remuestra se usa para
# los dos modelos a la vez; si el intervalo es todo negativo, el modelo pierde contra XGBoost de forma significativa).
logit_x_po = xgb_po["logit"].to_numpy()          # logit de default de XGBoost en el pool
logit_x_va = xgb_va["logit"].to_numpy()          # y en validación
ens_po, ens_va = lg_po.mean(0), lg_va.mean(0)                    # promedio de logits de las K redes (mean(0) = promedio por columna)
# Diccionario nombre -> logit de cada modelo; lg_po[0] es la primera red sola
scores_po = {"XGBoost": logit_x_po, "SNGP (1 red)": lg_po[0], f"Ensamble de {K} SNGP": ens_po,
             "XGB + Ensamble": 0.5 * (logit_x_po + ens_po), "XGB + SNGP (1 red)": 0.5 * (logit_x_po + lg_po[0])}
scores_va = {"XGBoost": logit_x_va, "SNGP (1 red)": lg_va[0], f"Ensamble de {K} SNGP": ens_va,
             "XGB + Ensamble": 0.5 * (logit_x_va + ens_va), "XGB + SNGP (1 red)": 0.5 * (logit_x_va + lg_va[0])}

filas = []                                       # una fila por modelo con sus seis AUC
for nombre in scores_po:
    fila = {"Modelo": nombre}
    # Misma cuenta en validación y en el pool: AUC total, solo thin-file (y[thin]) y del resto (~thin = "no thin")
    for tag, sc, y, thin in [("val", scores_va, y_va, thin_va), ("pool", scores_po, y_po, thin_po)]:
        fila[f"AUC {tag} total"] = roc_auc_score(y, sc[nombre])
        fila[f"AUC {tag} thin"] = roc_auc_score(y[thin], sc[nombre][thin])
        fila[f"AUC {tag} resto"] = roc_auc_score(y[~thin], sc[nombre][~thin])
    filas.append(fila)
print(pd.DataFrame(filas).set_index("Modelo").round(4).to_string())

# Bootstrap pareado en el pool thin-file: diferencia de AUC (modelo - XGBoost).
# "Pareado" = cada remuestra se usa para los dos modelos a la vez, así la diferencia no mezcla el ruido de
# remuestrear con la diferencia real entre modelos.
yt = y_po[thin_po]                               # TARGET de los thin-file del pool
rng = np.random.default_rng(SEED)
B = 1000                                         # cantidad de remuestras
# Cada remuestra: len(yt) posiciones sorteadas CON reposición entre 0 y len(yt)-1
idx_b = [rng.integers(0, len(yt), len(yt)) for _ in range(B)]    # mismas remuestras para todos
base = scores_po["XGBoost"][thin_po]             # el score de XGBoost, solo thin-file
for nombre in list(scores_po)[1:]:               # todos menos XGBoost (el primero)
    s = scores_po[nombre][thin_po]
    # Para cada remuestra i: AUC del modelo menos AUC de XGBoost sobre esas mismas filas
    dif = np.array([roc_auc_score(yt[i], s[i]) - roc_auc_score(yt[i], base[i]) for i in idx_b])
    lo, hi = np.percentile(dif, [2.5, 97.5])     # intervalo de 95%: percentiles 2.5 y 97.5 de las diferencias
    print(f"AUC thin {nombre} - XGBoost: {dif.mean():+.4f}  IC95% [{lo:+.4f}, {hi:+.4f}]")

# Diferencia de AUC thin en el pool contra XGBoost (IC 95%): SNGP (1 red) -0.0142 [-0.0204, -0.0076] | ensamble
# -0.0077 [-0.0127, -0.0020] | XGB + ensamble -0.0007 [-0.0032, +0.0022] | XGB + SNGP -0.0024 [-0.0054, +0.0010].
# Ningún modelo le gana a XGBoost. El ensamble cierra la mitad de la brecha de una red sola, pero sigue abajo de forma
# significativa: es el "−0,0077, IC [−0,013; −0,002]" de la sección 4.1 (en el 04 sale lo mismo con 2000 remuestras).
# Las mezclas empatan (el intervalo incluye el 0): sumarle la red a XGBoost no agrega nada, porque los dos ordenan
# casi igual (correlación de rangos de 0,94, Figura 4, en el 10).
# Ojo: la "SNGP (1 red)" de acá es la red 0 del ensamble, no la SNGP del 05 (que tiene otra semilla), por eso su AUC
# en el pool (0.7062) no es el 0.7091 de la Tabla 1.

                    AUC val total  AUC val thin  AUC val resto  AUC pool total  AUC pool thin  AUC pool resto
Modelo                                                                                                       
XGBoost                    0.7619        0.7251         0.7666          0.7523         0.7203          0.7560
SNGP (1 red)               0.7567        0.7175         0.7621          0.7460         0.7062          0.7510
Ensamble de 5 SNGP         0.7587        0.7216         0.7636          0.7490         0.7126          0.7535
XGB + Ensamble             0.7630        0.7269         0.7677          0.7533         0.7196          0.7574
XGB + SNGP (1 red)         0.7629        0.7261         0.7678          0.7527         0.7180          0.7568
AUC thin SNGP (1 red) - XGBoost: -0.0142  IC95% [-0.0204, -0.0076]
AUC thin Ensamble de 5 SNGP - XGBoost: -0.0077  IC95% [-0.0127, -0.0020]
AUC thin XGB + Ensamble - XGBoost: -0.0007  IC95% [-0.0032, +0.0022]
AUC thin XGB + SNGP (

In [4]:
# Bloque 4. El experimento conformal sobre thin-file con estos modelos, igual que en el 04: calibración por segmento,
# mitad del pool para calibrar y mitad para probar, 20 particiones. Si algún modelo hubiera ordenado mejor en el
# bloque anterior, acá se vería como más aprobados al mismo alfa.
ALFAS = [0.03, 0.05, 0.07, 0.09]                 # niveles de FDR: proporción máxima de morosos entre los aprobados
R = 20                                           # cantidad de particiones al azar del pool
y_i = y_po.astype(int)                           # TARGET como enteros 0/1
estrato = 2 * thin_po.astype(int) + y_i          # 0 resto pagador, 1 resto moroso, 2 thin pagador, 3 thin moroso
filas = []
for r in range(R):
    # Mitad para calibrar y mitad para probar; stratify mantiene la misma mezcla de los 4 grupos en cada mitad
    i_cal, i_test = train_test_split(np.arange(len(y_i)), test_size=0.5, stratify=estrato, random_state=r)
    rng2 = np.random.default_rng(r)              # para los desempates de los p-valores
    c, t = i_cal[thin_po[i_cal]], i_test[thin_po[i_test]]        # solo thin-file: calibración por segmento
    for nombre, lg in scores_po.items():
        mu = 1 - 1 / (1 + np.exp(-lg))                           # prob. de buen pago
        for alfa in ALFAS:
            # conformal_select(mu_cal, y_cal, mu_test, alfa, rng): True = aprobado
            a = conformal_select(mu[c], y_i[c], mu[t], alfa, rng2)
            # tasa = fracción aprobada; fdr = morosos aprobados / aprobados (max(.., 1) evita dividir por cero)
            filas.append({"r": r, "Modelo": nombre, "alfa": alfa, "tasa": a.mean(),
                          "fdr": (a & (y_i[t] == 1)).sum() / max(a.sum(), 1)})
res = pd.DataFrame(filas)
# Promedio de las 20 particiones: filas = modelo, columnas = alfa (reindex respeta el orden de scores_po)
tab = res.pivot_table(index="Modelo", columns="alfa", values="tasa").reindex(list(scores_po)).round(3)
print("Tasa de aprobación thin-file (por segmento):")
print(tab.to_string())
dif = tab - tab.loc["XGBoost"]                   # a cada fila le restamos la de XGBoost
print("\nDiferencia contra XGBoost (puntos de aprobación):")
print(dif.round(3).to_string())
res.to_csv("outputs/ensamble_resultados.csv", index=False)

# Aprobación thin-file a alfa 3 / 5 / 7 / 9%: XGBoost 0.289 / 0.544 / 0.804 / 0.956 | ensamble 0.267 / 0.542 / 0.786 /
# 0.954 | XGB + ensamble 0.288 / 0.538 / 0.798 / 0.956. Ningún modelo aprueba más que XGBoost en ningún alfa (coinciden
# con los números del 04: son las mismas particiones del pool). El ensamble mejora a una red sola sobre todo con
# alfa 3% (0.267 contra 0.237), que es donde el orden de los mejores candidatos importa más.

Tasa de aprobación thin-file (por segmento):
alfa                 0.03   0.05   0.07   0.09
Modelo                                        
XGBoost             0.289  0.544  0.804  0.956
SNGP (1 red)        0.237  0.516  0.787  0.953
Ensamble de 5 SNGP  0.267  0.542  0.786  0.954
XGB + Ensamble      0.288  0.538  0.798  0.956
XGB + SNGP (1 red)  0.279  0.542  0.793  0.955

Diferencia contra XGBoost (puntos de aprobación):
alfa                 0.03   0.05   0.07   0.09
Modelo                                        
XGBoost             0.000  0.000  0.000  0.000
SNGP (1 red)       -0.052 -0.028 -0.017 -0.003
Ensamble de 5 SNGP -0.022 -0.002 -0.018 -0.002
XGB + Ensamble     -0.001 -0.006 -0.006  0.000
XGB + SNGP (1 red) -0.010 -0.002 -0.011 -0.001


In [5]:
# Bloque 5. Diagnóstico de la distancia (sección 4.1 y párrafo de los terciles del informe).
# Primero, el cociente de medianas thin / resto con las dos medidas de incertidumbre del ensamble: la varianza de la
# capa GP (promedio de las 5 redes) y la discrepancia entre redes (varianza de los 5 logits). Mayor que 1 = la medida
# ve a los thin-file como más "lejanos" que el resto.
# Después, dentro de los thin-file del pool, armamos tres grupos (terciles) según cada medida y miramos el AUC de cada
# modelo en cada grupo. Si una medida de incertidumbre sirve para algo, el modelo debería ordenar PEOR donde la
# incertidumbre es alta. Es un diagnóstico descriptivo: cada grupo tiene entre 300 y 550 morosos y el AUC por grupo es
# ruidoso. Además los grupos tienen tasas de mora distintas, y eso también mueve el AUC.
var_gp = vr_po.mean(0)                                           # varianza GP media de las K redes
var_dis = lg_po.var(0)                                           # discrepancia entre redes (varianza de los logits)
# Cociente de medianas thin / resto: > 1 = la medida ve a los thin-file como más "lejanos" que el resto
print("mediana de la varianza GP, thin / resto:", round(float(np.median(var_gp[thin_po]) / np.median(var_gp[~thin_po])), 2))
print("mediana de la discrepancia, thin / resto:", round(float(np.median(var_dis[thin_po]) / np.median(var_dis[~thin_po])), 2))

for nombre_var, v in [("varianza GP", var_gp), ("discrepancia entre redes", var_dis)]:
    q = np.quantile(v[thin_po], [1 / 3, 2 / 3])                  # los dos cortes de terciles, solo con thin-file
    grupo = np.digitize(v[thin_po], q)                           # 0 = baja, 1 = media, 2 = alta
    filas = []
    for g, etiqueta in enumerate(["baja", "media", "alta"]):
        m = grupo == g                                           # True = el thin-file cae en este tercil
        fila = {f"{nombre_var}": etiqueta, "n": int(m.sum()), "morosos": int(yt[m].sum())}
        for nombre in ["XGBoost", f"Ensamble de {K} SNGP", "XGB + Ensamble"]:
            fila[nombre] = round(roc_auc_score(yt[m], scores_po[nombre][thin_po][m]), 4)   # AUC dentro del tercil
        filas.append(fila)
    print(pd.DataFrame(filas).to_string(index=False))

# Cociente thin / resto: varianza GP 1.01 y discrepancia 1.19. Otra vez, los thin-file no están más lejos (son dos de
# los cinco cocientes de la sección 4.1).
# AUC de XGBoost por tercil de varianza GP (baja / media / alta): 0.685 / 0.705 / 0.760. SUBE con la varianza, o sea que
# la varianza GP no marca dónde falla el modelo.
# AUC de XGBoost por tercil de discrepancia: 0.741 / 0.707 / 0.694. BAJA con la discrepancia: el desacuerdo entre redes
# sí marca dónde el modelo es menos confiable (una diferencia de ~2 errores estándar: sugerente, no concluyente).
# Son los números del párrafo de los terciles del informe y de la Lección 3. En los seis grupos el ensamble queda
# abajo de XGBoost.

mediana de la varianza GP, thin / resto: 1.01
mediana de la discrepancia, thin / resto: 1.19
varianza GP    n  morosos  XGBoost  Ensamble de 5 SNGP  XGB + Ensamble
       baja 4402      552   0.6846              0.6767          0.6840
      media 4402      450   0.7054              0.7005          0.7064
       alta 4402      335   0.7603              0.7458          0.7573
discrepancia entre redes    n  morosos  XGBoost  Ensamble de 5 SNGP  XGB + Ensamble
                    baja 4402      539   0.7409              0.7367          0.7414
                   media 4402      439   0.7072              0.7008          0.7071
                    alta 4402      359   0.6936              0.6776          0.6899
